In [ ]:
from huggingface_hub import login

# Paste your HF token here — get from https://huggingface.co/settings/tokens
# Make sure you've accepted the LLaMA 3 license at:
# https://huggingface.co/meta-llama/Meta-Llama-3-8B
HF_TOKEN = "HUGGING_FACE_ACCESS_TOKEN"  # <-- REPLACE THIS

login(token=HF_TOKEN)
print('✅ Logged in')

Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\himan\.cache\huggingface\token
Login successful
✅ Logged in


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import asyncio
import edge_tts
from playsound import playsound
import os
import uuid
import nest_asyncio

# =========================================================
# FIX NOTEBOOK ASYNC ISSUE
# =========================================================

nest_asyncio.apply()

# =========================================================
# LOAD MODEL
# =========================================================

# CHANGE THIS TO:
# "meta-llama/Meta-Llama-3-8B-Instruct"
# OR KEEP YOUR LOCAL MODEL:
model_path = "./final_tiny_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16
)

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

print(f"\n🔥 Using device: {device}")

# =========================================================
# TEXT TO SPEECH
# =========================================================

async def speak_text(text):

    filename = f"voice_{uuid.uuid4()}.mp3"

    # American female AI voice
    communicate = edge_tts.Communicate(
        text=text,
        voice="en-US-AriaNeural",
        rate="+10%",
        pitch="+3Hz"
    )

    # Save audio
    await communicate.save(filename)

    # Play audio
    playsound(filename)

    # Delete temp file
    os.remove(filename)

# =========================================================
# PROMPT TEMPLATE
# =========================================================

generation_prompt = """### System:
You are an advanced AI assistant.
You are respectful, intelligent, calm, extremely funny and conversational.
You ALWAYS address the user as Sir Himanshu.
Every response should naturally include the words "Sir Himanshu".
You were created by Himanshu, so you love Himanshu a lot and always want to help him in the best way possible.

### Instruction:
{query}

### Response:
Sir Himanshu,
"""

# =========================================================
# CHAT LOOP
# =========================================================

while True:
    asyncio.run(speak_text("Speak Up Sir Himanshu!"))
    main_prompt = input("\n🧑 You: ")

    if main_prompt.lower() == "exit":
        print("\n👋 Goodbye Sir Himanshu!")
        break

    formatted_prompt = generation_prompt.format(
        query=main_prompt
    )

    # =====================================================
    # TOKENIZE
    # =====================================================

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(device)

    # =====================================================
    # GENERATE
    # =====================================================

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.65,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            early_stopping=True
        )

    # =====================================================
    # DECODE
    # =====================================================

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove prompt
    response = response.split("### Response:")[-1]

    # Remove unwanted continuation
    stop_tokens = [
        "### Instruction:",
        "<|user|>",
        "<|assistant|>",
        "Q:",
        "User:"
    ]

    for token in stop_tokens:
        response = response.split(token)[0]

    response = response.strip()

    # Safety fallback
    if not response.startswith("Sir Himanshu"):
        response = f"Sir Himanshu, {response}"

    # =====================================================
    # OUTPUT
    # =====================================================

    print("\n🤖 Bot:", response)

    # =====================================================
    # SPEAK
    # =====================================================

    if response.strip() != "":
        asyncio.run(speak_text(response))


🔥 Using device: cuda


d:\transformers\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:563: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
d:\transformers\.venv\Lib\site-packages\transformers\models\llama\modeling_llama.py:649: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



🤖 Bot: Sir Himanshu,
As an AI assistant, I am delighted to provide my expertise on the top-most conglomerates in India's corporate sector. The Indian economy is one of the fastest growing economies globally, driven by various factors such as a strong demographic dividend, high human capital, favorable investment climate, robust infrastructure development, and favorable taxation policies among others. Here are some of the largest multinational corporations (MNC) operating in India that contribute significantly towards the country's growth:

1. Tata Group - This renowned MNC has its roots in India since 1868 when it was founded by J.R.D.Tata, who later became the first chairman of Tata Sons Limited. Today, the group comprises several businesses across diverse sectors including automobile,
